# Clasificador de Reclamos - TF-IDF + XGBoost

Pipeline de entrenamiento para clasificar reclamos de consumidores en 8 macro-categorias
usando TF-IDF para vectorizacion de texto y XGBoost como clasificador.

**Pasos:**
1. Carga de datos
2. Analisis exploratorio (EDA)
3. Preparacion de datos (TF-IDF)
4. Entrenamiento del modelo base
5. Optimizacion de hiperparametros
6. Evaluacion del modelo (Precision, Recall, F1-Score)
7. Guardado del modelo

In [ ]:
# Instalar dependencias (descomentar si es necesario)
# !pip install pandas numpy scikit-learn xgboost joblib matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import joblib
import xgboost as xgb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

print('Librerias cargadas')

In [ ]:
# === CONFIGURACION ===

DATA_PATH = 'clasificador-reclamos-backend/data/quejas_limpias.csv'
MODEL_PATH = 'clasificador-reclamos-backend/models/model.pkl'
METRICS_PATH = 'clasificador-reclamos-backend/models/metrics.json'

TARGET = 'CATEGORIA'
TEXT_COL = 'texto'
TEST_SIZE = 0.2
RANDOM_STATE = 42

## 1. Carga de Datos

In [ ]:
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')

print(f'Dataset: {df.shape[0]:,} registros, {df.shape[1]} columnas')
print(f'Columnas: {df.columns.tolist()}')
print(f'Nulos: {df.isnull().sum().sum()}')
df.head()

## 2. Analisis Exploratorio (EDA)

In [ ]:
# Distribucion del target
print(f'Categorias: {df[TARGET].nunique()}\n')

counts = df[TARGET].value_counts()
for cat, count in counts.items():
    pct = count / len(df) * 100
    print(f'  {cat:<35} {count:>6,}  ({pct:>5.1f}%)')

In [ ]:
# Grafico de distribucion del target
fig, ax = plt.subplots(figsize=(10, 5))
counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Distribucion de {TARGET}')
ax.set_xlabel('Cantidad de registros')
for i, (val, count) in enumerate(counts.items()):
    ax.text(count + 50, i, f'{count:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Longitud del texto
df['text_len'] = df[TEXT_COL].str.len()

print('Longitud del texto:')
print(f'  Min: {df["text_len"].min()}, Max: {df["text_len"].max()}, Media: {df["text_len"].mean():.0f}')

fig, ax = plt.subplots(figsize=(8, 4))
df['text_len'].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribucion de longitud de texto')
ax.set_xlabel('Caracteres')
plt.tight_layout()
plt.show()

df.drop(columns=['text_len'], inplace=True)

## 3. Preparacion de Datos

In [ ]:
# Codificar target
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df[TARGET])

print(f'Clases: {target_encoder.classes_.tolist()}')
print(f'Codificacion: {dict(zip(target_encoder.classes_, range(len(target_encoder.classes_))))}')

In [ ]:
# TF-IDF Vectorizacion
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    min_df=2,
    sublinear_tf=True,
)

X = tfidf.fit_transform(df[TEXT_COL])

print(f'Matriz TF-IDF: {X.shape[0]:,} documentos x {X.shape[1]:,} features')
print(f'Densidad: {X.nnz / (X.shape[0] * X.shape[1]) * 100:.2f}%')

In [ ]:
# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

# Pesos de muestra para balanceo de clases
sample_weights = compute_sample_weight('balanced', y_train)
print('Sample weights calculados para balanceo de clases.')

## 4. Entrenamiento del Modelo Base

In [ ]:
base_model = xgb.XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

base_model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

base_score = base_model.score(X_test, y_test)
print(f'Accuracy del modelo base: {base_score:.4f}')

## 5. Optimizacion de Hiperparametros

In [ ]:
param_distributions = {
    'n_estimators': [200, 300, 500],
    'max_depth': [4, 5, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [3, 5, 7],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0.5, 1.0, 1.5],
}

search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_distributions=param_distributions,
    n_iter=20,
    scoring='f1_weighted',
    cv=3,
    random_state=RANDOM_STATE,
    verbose=1,
    n_jobs=-1,
)

search.fit(X_train, y_train, sample_weight=sample_weights)

best_model = search.best_estimator_
best_params = search.best_params_

print(f'Mejor F1 (CV): {search.best_score_:.4f}')
print(f'Mejores parametros: {best_params}')

In [ ]:
# Validacion cruzada del mejor modelo
cv_scores = cross_val_score(
    best_model, X_train, y_train,
    cv=5,
    scoring='f1_weighted',
)

print(f'F1 weighted (5-fold CV): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
print(f'Scores individuales: {[f"{s:.4f}" for s in cv_scores]}')

## 6. Evaluacion del Modelo

Metricas clave:
- **Precision**: De los que predije como clase X, cuantos realmente lo son?
- **Recall**: De todos los que son clase X, cuantos identifique correctamente?
- **F1-Score**: Media armonica de Precision y Recall. Metrica principal para clasificacion desbalanceada.

In [ ]:
# Predicciones sobre test
y_pred = best_model.predict(X_test)

# Classification Report completo
print('=' * 70)
print('CLASSIFICATION REPORT')
print('=' * 70)
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

In [ ]:
# Matriz de Confusion
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_encoder.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, values_format='d')
ax.set_title('Matriz de Confusion')
plt.tight_layout()
plt.show()

In [ ]:
# Analisis de Precision, Recall y F1 por clase
report = classification_report(y_test, y_pred, target_names=target_encoder.classes_, output_dict=True)

metrics_df = pd.DataFrame(report).T
metrics_df = metrics_df[metrics_df.index.isin(target_encoder.classes_)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, metric in enumerate(['precision', 'recall', 'f1-score']):
    ax = axes[i]
    values = metrics_df[metric].sort_values()
    colors = ['#e74c3c' if v < 0.5 else '#f39c12' if v < 0.7 else '#27ae60' for v in values]
    values.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(metric.upper())
    ax.set_xlim(0, 1)
    ax.axvline(x=0.7, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Metricas por Categoria', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Top features mas importantes del TF-IDF
feature_names = tfidf.get_feature_names_out()
importance = best_model.feature_importances_

top_n = 20
top_idx = np.argsort(importance)[-top_n:]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(top_n), importance[top_idx], color='steelblue')
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in top_idx])
ax.set_title(f'Top {top_n} Features (TF-IDF)')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

## 7. Guardado del Modelo y Metricas

In [ ]:
# Guardar metricas
metrics = {
    'classification_report': report,
    'confusion_matrix': cm.tolist(),
    'best_params': best_params,
    'cv_f1_mean': float(cv_scores.mean()),
    'cv_f1_std': float(cv_scores.std()),
    'categories': target_encoder.classes_.tolist(),
}

os.makedirs(os.path.dirname(METRICS_PATH), exist_ok=True)
with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f'Metricas guardadas en: {METRICS_PATH}')

In [ ]:
# Empaquetar modelo + artefactos
artifacts = {
    'model': best_model,
    'tfidf': tfidf,
    'target_encoder': target_encoder,
    'categories': target_encoder.classes_.tolist(),
}

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(artifacts, MODEL_PATH)
print(f'Modelo guardado en: {MODEL_PATH}')

# Verificar serializacion
loaded = joblib.load(MODEL_PATH)
assert loaded['model'] is not None
assert loaded['tfidf'] is not None
print('Modelo serializado y verificado correctamente.')

## 8. Prueba de Prediccion

In [ ]:
test_texts = [
    'me cobraron de mas en una tienda departamental',
    'no me quieren entregar el producto que compre por internet',
    'el refrigerador tiene defectos de fabrica y no aceptan la garantia',
    'quiero cancelar mi contrato y no me dejan',
    'me depositaron menos de lo que correspondia',
]

loaded = joblib.load(MODEL_PATH)
model = loaded['model']
tfidf_loaded = loaded['tfidf']
te = loaded['target_encoder']

for text in test_texts:
    X_new = tfidf_loaded.transform([text])
    proba = model.predict_proba(X_new)[0]
    pred_idx = int(np.argmax(proba))
    pred_label = te.inverse_transform([pred_idx])[0]
    confidence = float(proba[pred_idx])

    print(f'Texto: "{text}"')
    print(f'  -> {pred_label} (confianza: {confidence:.2%})')
    print()

In [ ]:
print('=' * 60)
print('  PIPELINE COMPLETADO')
print('=' * 60)
print(f'Modelo:   {MODEL_PATH}')
print(f'Metricas: {METRICS_PATH}')
print(f'Para iniciar la API:')
print(f'  cd clasificador-reclamos-backend')
print(f'  uvicorn server:app --reload')